# 132543: sensitivity-driven sparse-grid scan of an NSTX pedestal

**Purpose of this notebook.** A readable walkthrough of the scan pipeline, for
discussion. It shows where the sparse grid enters, what it scans over, and what
one function evaluation costs.

**Pipeline.**

```
   [TPED]                                       [sg_lib]
   experimental discharge (EQDSK + pfile)
        |
        |  mtanh fit of the pedestal, per species
        v
   nominal fit parameters  ------------------>  scan axes are SCALE FACTORS
        ^                                       on these fit parameters
        |                                            |
        |                                       SparseScanSession.ask()
        |                                            |  batch of grid points
        |<-------------------------------------------+
        |
        |  apply scale factors -> perturbed profiles
        |  CHEASE-BS reconstruction (bootstrap-consistent equilibrium)
        |  acceptance gate (Ip, q at the analysis radii)
        v
   per-point EQDSK + iterdb + GENE parameters file
        |
        |  GENE linear run -> growth rate at one k_y
        v
   scalar QoI  ------------------------------>  SparseScanSession.tell()
                                                    |
                                               refinement: next batch,
                                               or termination
```

`sg_lib` is the sensitivity-driven dimension-adaptive sparse-grid library
(Farcas et al., JCP 410:109394). `SparseScanSession` is our ask/tell wrapper
around it: it maps the unit hypercube to physical bounds, persists state as an
ordered (point, value) list, and rebuilds the sg_lib objects by deterministic
replay so the loop survives interpreter restarts and queue gaps.

**Status of this run.** No GENE. The submitter is replaced by one that writes
the parameters file and stops, and the grid is advanced on **fabricated** QoI
values so it emits more than the single seed point. Everything upstream of the
GENE call is the production path. The refinement trajectory and the surrogate
below are therefore machinery demonstrations, not physics.

Needs NERSC: CHEASE-BS is a compiled binary.

In [ ]:
# --------------------------------------------------------------- inputs
BASE_PARAMETERS = "/global/homes/j/joeschm/simulators/GENE3/NSTX_132543_omne0.8_omt0.8_KBM_r0.7/parameters"
SEED_DIRPATH    = "/global/homes/j/joeschm/data/ST_research/NSTXU_discharges/132543"
SEED_ITERDB     = "/global/homes/j/joeschm/data/ST_research/NSTXU_discharges/132543/NSTX132543.iterdb"
SEED_GFILE      = "/global/homes/j/joeschm/data/ST_research/NSTXU_discharges/132543/g132543.00700"

OUTROOT      = "tmp_runs"
KY           = 0.05          # single k_y; the QoI is gamma at this k_y
STEPS        = 3             # refinement steps to walk (batch 0 is ONE point)
MAX_POINTS   = 12            # hard cap: every point is a CHEASE-BS run
EQDSK_PREFIX = "g132543"
RECON_DIR    = None

# CHEASE-BS settings. max_iter is a physics setting, not a cost knob: the solver
# under-relaxes (bootstrap_mix 0.1, istar_mix 0.05), so a low cap returns an
# equilibrium still carrying the baseline current profile.
CHEASEBS_OVERRIDES = {
    "max_iter": 25,
    "tol_bs": 1e-4, "tol_q": 1e-4, "tol_ip_rel": 0.02,
    "bootstrap_mix": 0.1, "istar_mix": 0.05,
    "plot_errors": True,
}

In [ ]:
import os, sys, json
from datetime import datetime

sys.path.insert(0, os.path.abspath("."))
import pilot_helpers as ph

WORKDIR = os.path.abspath(os.path.join(OUTROOT, datetime.now().strftime("%Y%m%d_%H-%M-%S")))
os.makedirs(WORKDIR, exist_ok=True)
with open(os.path.join(WORKDIR, "WARNING-DUMMY-SESSION.txt"), "w") as f:
    f.write("session.json here was advanced with fabricated QoI values. "
            "Do not resume a real campaign from this directory.\n")

ph.check_inputs({"BASE_PARAMETERS": BASE_PARAMETERS,
                 "SEED_ITERDB": SEED_ITERDB, "SEED_GFILE": SEED_GFILE})
print("workdir:", WORKDIR)

## 1. What the grid scans over

The scan axes are **dimensionless scale factors on the nominal mtanh fit**, not
absolute physical values. One fit per discharge, made once and reused at every
node, so the same scan coordinate means the same physics at every node.

In [ ]:
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh
from TPED.projects.GENE_pipelines.src.point_reconstruction import (
    PILOT_BOUNDS_132543 as BOUNDS, MTANH_AXES, MTANH_FIT_KWARGS,
    nominal_fits, _apply_axes, is_equilibrium_axis)

discharge = ph.seed_from_iterdb(SEED_ITERDB, SEED_GFILE, WORKDIR)
phys0 = DischargePhysics(discharge)

print("scan axes (scale factors on the nominal fit):\n")
print(f"{'axis':<17} {'var':<4} {'fit parameter':<15} {'range':<14} meaning")
for name, (lo, hi) in BOUNDS.items():
    var, kwarg = MTANH_AXES[name]
    meaning = {"scale_height": "pedestal step amplitude",
               "scale_width":  "pedestal width"}[kwarg]
    print(f"{name:<17} {var:<4} {kwarg:<15} [{lo}, {hi}]{'':<4} {meaning}")

print("\nnominal fit these scale (fit quality first):")
for var in sorted({v for v, _ in MTANH_AXES.values()}):
    _, record = fit_mtanh(phys0.ds, var, **MTANH_FIT_KWARGS)
    pars = "  ".join(f"{k}={v:.4g}" for k, v in record["fit_params"].items())
    print(f"  {var:<4} rms/range = {record['rms_relative']:.4f}   {pars}")

The corners of the box, so the axes are visible as profiles. This is the whole
physical content of the scan space.

In [ ]:
import matplotlib.pyplot as plt

fits = nominal_fits(phys0, set(BOUNDS))
corners = [("nominal", {})]
for axis in ("Te_ped_scale", "Te_width_scale"):
    lo, hi = BOUNDS[axis]
    corners += [(f"{axis} {lo}", {axis: lo}), (f"{axis} {hi}", {axis: hi})]

fig, ax = plt.subplots(figsize=(7, 4))
for label, point in corners:
    p = _apply_axes(phys0, point, fits=fits) if point else phys0
    ax.plot(p.rhot, p.Te, lw=1.5, label=label)
ax.set_xlim(0.5, 1.0); ax.set_xlabel("rho_tor"); ax.set_ylabel("Te (eV)")
ax.set_title("Te pedestal at the box corners"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. The sparse-grid session

`SparseScanSession` takes the axis bounds and nothing else about the physics.
`ask()` returns the next batch of surplus points in physical units; `tell()`
takes one scalar per point, in the same order. One session drives one scalar
QoI, so growth rate and frequency are separate sessions.

In [ ]:
from TPED.projects.GENE_pipelines.src.sparse_scan_driver import SparseScanSession

sampler = SparseScanSession(BOUNDS, state_path=os.path.join(WORKDIR, "session.json"))

print(f"dimension  {sampler.dim}")
print(f"tol        {sampler.tol}")
print(f"max_level  {sampler.max_level}")
print(f"state      {sampler.state_path}")
print(f"\nfirst batch from ask() -- {len(sampler.ask())} point(s):")
for i, point in enumerate(sampler.ask()):
    print("  ", i, {k: round(v, 4) for k, v in point.items()})

## 3. One function evaluation

Each point from `ask()` goes through: apply the scale factors to the nominal fit
-> perturbed profiles (quasineutrality enforced) -> CHEASE-BS reconstruction
against the source EFIT boundary -> acceptance gate on Ip and on q at the two
radii the GENE runs sit at -> per-point EQDSK and iterdb -> one GENE parameters
file. A rejected point is a node the grid asked for and cannot have, which is
the interesting failure mode of a prescribed-node method.

In [ ]:
from TPED.projects.discharge_tools.src.cheasebs_runner import CheasebsAcceptance
from TPED.projects.GENE_pipelines.src.scan_campaign import (
    GeneScanCampaign, QoISpec, HARVESTED, REJECTED)

GENE_RADII = (0.736, 0.825)      # r_0.736 (q=4), r_0.825 (q=5)
AXIS_SHORT = {"Te_ped_scale": "Te", "ne_ped_scale": "ne", "Te_width_scale": "wTe"}

campaign = GeneScanCampaign(
    sampler=sampler,
    base_discharge=discharge,
    qoi=QoISpec(quantity="gamma", reduction="at_ky", ky=KY),
    workdir=WORKDIR,
    base_parameters=os.path.abspath(BASE_PARAMETERS),
    acceptance=CheasebsAcceptance.production(analysis_radii=GENE_RADII),
    ky_scanlist=[KY],
    recon_dir=RECON_DIR,
    cheasebs_overrides=CHEASEBS_OVERRIDES,
)
campaign.submitter = ph.write_parameters_only(campaign)   # writes parameters, does not submit
print("campaign ready")

## 4. The ask/tell loop

Slow cell: one CHEASE-BS run per point. In production `tell()` receives the
harvested growth rates; here it receives fabricated values so the grid refines.

In [ ]:
proposed = 0
for step in range(STEPS):
    accepted = campaign.propose()
    batch = campaign.ledger.batch(step)
    proposed += len(batch)
    print(f"\n=== step {step}: {len(batch)} node(s) asked, {len(accepted)} accepted")

    ph.retag_eqdsks(campaign, step, EQDSK_PREFIX, AXIS_SHORT)
    campaign.submit_pending()

    for e in batch:
        pt = "  ".join(f"{AXIS_SHORT.get(k, k)}={v:.3f}" for k, v in sorted(e.point.items()))
        a = e.acceptance or {}
        ip = a.get("ip_error_rel")
        ip = f"{ip:.3%}" if isinstance(ip, (int, float)) else "n/a"
        if e.status == REJECTED:
            print(f"  {e.point_id}  {pt}  REJECTED: {e.note}")
        else:
            print(f"  {e.point_id}  {pt}  cheaseBS iters={a.get('cheasebs_iterations')}"
                  f"  Ip err={ip}"
                  f"  -> {os.path.basename(e.rundir or '(no rundir)')}")

    stuck = [e for e in batch if not e.rundir and e.eqdsk]
    if stuck:
        print(f"  STOP: {len(stuck)} point(s) wrote no parameters file"); break
    if proposed >= MAX_POINTS:
        print(f"  STOP: point cap {MAX_POINTS}"); break
    if len(accepted) != len(batch):
        print("  STOP: batch incomplete, so the grid cannot advance"); break

    for e in batch:
        campaign.ledger.update(e.point_id, status=HARVESTED,
                               qoi=ph.dummy_qoi(e.point), note="DUMMY_QOI")
    campaign.tell_batch(step)
    print(f"  told the grid {len(batch)} fabricated value(s)")

## 5. Did every node evaluate?

The question a prescribed-node method makes sharp: a node we cannot deliver is
a hole in the interpolant, not a noisy label. Two ways to lose one here, and
both are visible below: CHEASE-BS fails to converge, or the acceptance gate
rejects the equilibrium it produced.

In [ ]:
ph.convergence_report(campaign)

problems = ph.verify_parameters(campaign, is_equilibrium_axis)
print("\nparameters-file check:",
      "clean" if not problems else f"{len(problems)} problem(s): {problems}")

## 6. Grid state

`axis_refinement_levels()` is the axis-importance readout: an axis left at level
1 was never refined, i.e. pruned. `surrogate()` returns the interpolant over the
multi-indices whose evaluations are in hand, callable mid-scan.

In [ ]:
print(f"evaluations used   {sampler.n_evals}")
print(f"finished           {sampler.finished}")
print(f"refinement levels  {sampler.axis_refinement_levels()}")

print("\nevaluations, in the order the grid asked for them:")
print(f"{'#':>3}  " + "  ".join(f"{AXIS_SHORT.get(n, n):>7}" for n in sampler.names) + "      QoI")
for i, (std, value) in enumerate(sampler._evals):
    phys = [lo + s * (hi - lo) for s, (lo, hi) in zip(std, sampler.params.values())]
    print(f"{i:>3}  " + "  ".join(f"{v:>7.3f}" for v in phys) + f"   {value:>10.4g}")

In [ ]:
# 1-D slices of the interpolant through the nominal point. Shape is meaningless
# here (fabricated QoI) -- this is the interpolant call, not a result.
import numpy as np

try:
    f = sampler.surrogate()
    nominal = {n: 1.0 for n in sampler.names}
    fig, axes = plt.subplots(1, sampler.dim, figsize=(4 * sampler.dim, 3), sharey=True)
    for ax, name in zip(np.atleast_1d(axes), sampler.names):
        lo, hi = sampler.params[name]
        xs = np.linspace(lo, hi, 60)
        ys = [f([xs[k] if n == name else nominal[n] for n in sampler.names])
              for k in range(len(xs))]
        ax.plot(xs, ys); ax.set_xlabel(AXIS_SHORT.get(name, name))
    np.atleast_1d(axes)[0].set_ylabel("interpolant (fabricated QoI)")
    plt.tight_layout(); plt.show()
except RuntimeError as exc:
    print("no interpolant yet:", exc)

## 7. What one node hands to GENE

A run directory holding a GENE run and nothing else: the parameters file, its
own EQDSK, its own iterdb. Reconstruction artifacts stay out, so the directory
can be tarred and moved.

In [ ]:
written = [e for e in campaign.ledger.entries.values() if e.rundir]
if written:
    e = written[0]
    print(e.rundir)
    for f in sorted(os.listdir(e.rundir)):
        print("   ", f)
    print("\n" + open(os.path.join(e.rundir, "parameters")).read())
else:
    print("nothing written")

In [ ]:
summary = os.path.join(WORKDIR, "walkthrough_summary.json")
with open(summary, "w") as f:
    json.dump({"workdir": WORKDIR, "axes": BOUNDS, "ky": KY,
               "analysis_radii": list(GENE_RADII),
               "qoi_values_are_fabricated": True,
               "n_evals": sampler.n_evals,
               "refinement_levels": sampler.axis_refinement_levels(),
               "problems": problems,
               "points": [{"point_id": e.point_id, "batch": e.batch,
                           "point": e.point, "status": e.status,
                           "rundir": e.rundir, "note": e.note}
                          for e in campaign.ledger.entries.values()]}, f, indent=1)
print(summary)